[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/00_siapkan_dataset.ipynb)

# 00 — Siapkan semua dataset buku di Google Drive
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*

Jalankan notebook ini **sekali** (Colab, *Runtime → Run all*; GPU tidak diperlukan). Semua dataset diunduh dari
**sumber resminya** langsung ke `MyDrive/Buku Visi Komputer Kesehatan/dataset/<nama_dataset>/`. Berkas yang sudah ada dan
lolos cek (ukuran, MD5 bila sumber menyediakan, atau jumlah berkas di dalam zip) **tidak diunduh ulang**, sehingga
notebook aman dijalankan berkali-kali (mis. setelah koneksi terputus).

Perkiraan total ruang Drive bila semua pilihan bawaan aktif: ± 2,6 GB (lihat tabel ringkasan di akhir).
Notebook bab (`bab01.ipynb`, dst.) memakai fungsi yang sama (`pastikan_dataset`), jadi menjalankan notebook ini lebih dulu
hanya mempercepat; bukan syarat.

> **Lisensi.** Mengunduh = menerima lisensi tiap sumber. DermaMNIST (CC BY-NC 4.0) dan Kvasir-SEG (riset/pendidikan)
> **tidak untuk penggunaan komersial**. Montgomery County CXR Set **tidak boleh didistribusikan ulang** dan nonaktif bawaan.
> Jangan membagikan folder `dataset/` secara publik; bagikan notebook, bukan datanya.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 1 Lingkungan dan pencatat waktu

In [ ]:
import contextlib, platform
WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter(); yield
    WAKTU[nama] = round(time.perf_counter() - t0, 1); print(f"[waktu] {nama}: {WAKTU[nama]} s")
smi = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout
print(smi if smi.strip() else "nvidia-smi: tidak ada GPU (tidak masalah untuk notebook ini)")
print("Python", platform.python_version())

## 2 Pilih dataset
Ubah `True`/`False` bila ruang Drive terbatas. Kolom komentar menunjukkan bab yang memakainya.

In [ ]:
MEDMNIST_BUKU = [  # (flag, ukuran px)  — bab pemakai
    ("pneumoniamnist", 28), ("pneumoniamnist", 64), ("pneumoniamnist", 128), ("pneumoniamnist", 224),  # Bab 1, 4, 12
    ("organamnist", 64),                                                                                  # Bab 1, 5
    ("breastmnist", 28), ("breastmnist", 128),                                                            # Bab 1, 3
    ("bloodmnist", 28), ("bloodmnist", 64),                                                               # Bab 1, 3, 12
    ("retinamnist", 28), ("retinamnist", 128), ("retinamnist", 224),                                      # Bab 1, 7
    ("octmnist", 64),                                                                                     # Bab 1, 5
    ("organmnist3d", 28), ("organmnist3d", 64),                                                           # Bab 10
    ("nodulemnist3d", 28), ("nodulemnist3d", 64),                                                         # Bab 10
]
PAKAI_DERMAMNIST = True     # Bab 5; lisensi CC BY-NC 4.0 (nonkomersial)
DERMA = [("dermamnist", 28), ("dermamnist", 64)]
PILIH = {
    "medmnist":   True,   # Zenodo 10519652, CC BY 4.0 (kecuali DermaMNIST)
    "malaria":    True,   # NIH/NLM, Bab 2 & 6 (353 MB, 27.558 PNG di dalam zip)
    "hc18":       True,   # Zenodo 1327317, CC BY 4.0, Bab 7 & 9 (± 177 MB)
    "kvasir_seg": True,   # Simula, riset/pendidikan, Bab 9 (46 MB)
    "bccd":       True,   # GitHub Shenggan/BCCD_Dataset, MIT, Bab 8 (± 8 MB)
    "montgomery": False,  # NLM, opsional; dilarang didistribusikan ulang — baca ReadMe dulu
}

## 3 Unduh atau verifikasi

In [ ]:
STATUS = []
def jalankan(nama, berkas=None):
    try:
        with catat_waktu(f"pastikan_{nama}"):
            pastikan_dataset(nama, berkas)
        STATUS.append((nama, "OK"))
    except Exception as e:   # satu sumber gagal tidak menghentikan yang lain
        print(f"!! {nama}: {type(e).__name__}: {e}"); STATUS.append((nama, f"GAGAL: {e}"))

if PILIH["medmnist"]:
    daftar = MEDMNIST_BUKU + (DERMA if PAKAI_DERMAMNIST else [])
    jalankan("medmnist", [berkas_medmnist(f, s) for f, s in daftar])
for nama in ("malaria", "hc18", "kvasir_seg", "bccd"):
    if PILIH[nama]: jalankan(nama)

## 4 (Opsional) Montgomery County CXR Set
NLM tidak menyediakan satu arsip; sel ini mengunduh 138 citra PNG + 2×138 masker paru per berkas ke disk lokal, lalu
mengemasnya menjadi `montgomery_set.zip` di Drive (cek: 414 PNG). Data klinis (`ClinicalReadings`) sengaja tidak diunduh.
Buku tidak memakai dataset ini sebagai jalur utama. **Larangan redistribusi**: simpan hanya di Drive pribadi.

In [ ]:
BASIS_MC = "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/"
SUB_MC = ["CXR_png/", "ManualMask/leftMask/", "ManualMask/rightMask/"]
def pastikan_montgomery():
    tujuan = DATASET_DIR / "montgomery" / "montgomery_set.zip"
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    if tujuan.exists():
        n = sum(1 for x in zipfile.ZipFile(tujuan).namelist() if x.endswith(".png"))
        if n == 414: print(f"Dataset montgomery ditemukan di Drive ({tujuan.stat().st_size/1e6:.0f} MB, {n} PNG)."); return
        print(f"montgomery_set.zip berisi {n} PNG (seharusnya 414); mengunduh ulang...")
    else:
        print("Dataset montgomery belum ada, mengunduh per berkas dari NLM...")
    lokal = LOKAL / "montgomery_unduh"
    for sub in SUB_MC:
        html = requests.get(BASIS_MC + sub + "index.html", timeout=60).text
        nama = sorted(set(re.findall(r"href='([^'/]+\.png)'", html)))
        print(f"  {sub}: {len(nama)} berkas")
        (lokal / sub).mkdir(parents=True, exist_ok=True)
        for fn in nama:
            p = lokal / sub / fn
            if not p.exists():
                r = requests.get(BASIS_MC + sub + fn, timeout=120); r.raise_for_status(); p.write_bytes(r.content)
    r = requests.get(BASIS_MC + "NLM-MontgomeryCXRSet-ReadMe.pdf", timeout=60); (lokal / "NLM-MontgomeryCXRSet-ReadMe.pdf").write_bytes(r.content)
    arsip = LOKAL / "montgomery_set.zip"
    with zipfile.ZipFile(arsip, "w", zipfile.ZIP_STORED) as z:
        for p in sorted(lokal.rglob("*")):
            if p.is_file(): z.write(p, p.relative_to(lokal))
    shutil.copyfile(arsip, tujuan); print("  tersimpan:", tujuan)

if PILIH["montgomery"]:
    with catat_waktu("pastikan_montgomery"):
        pastikan_montgomery()
else:
    print("Montgomery dilewati (PILIH['montgomery'] = False).")

## 5 Dataset yang memerlukan akun atau registrasi (hanya petunjuk)
Notebook ini **tidak** mengunduhnya otomatis dan **tidak menyimpan kunci API apa pun**.

| Dataset | Mengapa tidak otomatis | Pengganti di buku |
|---|---|---|
| APTOS 2019 Blindness Detection (Kaggle) | perlu akun Kaggle + menyetujui aturan kompetisi + token API | RetinaMNIST (Bab 7) |
| Chest X-Ray Images (Pneumonia), mirror Kaggle | perlu akun Kaggle; versi asli Kermany dkk. ada di Mendeley Data tanpa login, tetapi arsipnya ± 8,4 GB (CXR + OCT) dan lisensinya perlu dikonfirmasi | PneumoniaMNIST (Bab 4) |
| Kvasir-SEG (bila unduhan langsung diblokir) | Simula dapat meminta formulir | HC18 segmentasi (Bab 9) |

Bila Anda tetap ingin memakai Kaggle di Colab: simpan token di **Colab → ikon kunci (Secrets)** dengan nama
`KAGGLE_USERNAME` dan `KAGGLE_KEY`, lalu baca dengan `google.colab.userdata.get(...)` di sel Anda sendiri.
Jangan menuliskan token di notebook, jangan mengunggah `kaggle.json` ke folder yang dibagikan.

## 6 Ringkasan isi Drive

In [ ]:
import pandas as pd
baris = []
for d in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    berkas = [p for p in d.rglob("*") if p.is_file() and not p.name.endswith((".md5ok", ".isiok"))]
    isi_zip = 0
    for p in berkas:
        if p.suffix == ".zip":
            try: isi_zip += sum(1 for x in zipfile.ZipFile(p).namelist() if not x.endswith("/"))
            except zipfile.BadZipFile: pass
    baris.append({"dataset": d.name, "jumlah_berkas_di_drive": len(berkas),
                  "berkas_di_dalam_zip": isi_zip, "ukuran_MB": round(sum(p.stat().st_size for p in berkas) / 1e6, 1),
                  "terverifikasi_MD5": sum(1 for p in d.rglob("*.md5ok"))})
RINGKAS = pd.DataFrame(baris)
RINGKAS.loc[len(RINGKAS)] = {"dataset": "TOTAL", "jumlah_berkas_di_drive": RINGKAS.jumlah_berkas_di_drive.sum(),
                             "berkas_di_dalam_zip": RINGKAS.berkas_di_dalam_zip.sum(),
                             "ukuran_MB": round(RINGKAS.ukuran_MB.sum(), 1), "terverifikasi_MD5": RINGKAS.terverifikasi_MD5.sum()}
RINGKAS.to_csv(DATASET_DIR / "ringkasan_dataset.csv", index=False)
print("Status:", STATUS); print("Waktu (s):", WAKTU)
RINGKAS